# Imports

In [1]:
import os
import pickle
import re
import shutil
import sys
sys.path.append(os.path.dirname(os.getcwd()))
from itertools import product

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from matplotlib.backends.backend_pdf import PdfPages
from tools import load_npy, load_yaml_as_df, load_pkl, exist_metric, exist_stf_metric, inverse_stf_metrics, keep_split, is_full_group

plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
plt.rc('font', family='Arial')
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.size'] = 10

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# projection by STFT and Wavelet

## load data

In [2]:
root = '/data/home/Licheng/workspace/TSF-PCA/results_PCA/diff_trans_local'
exp_dirs = os.listdir(root)
exp_dirs = [os.path.join(root, exp_dir) for exp_dir in exp_dirs]

params = ['model', 'seq_len', 'pred_len', 'data_id', 'learning_rate', 'rec_lambda', 'auxi_lambda', 'pca_dim', 'reinit', 'use_weights', 'rank_ratio', 'auxi_loss', 'batch_size', 'lradj', 'patience', 'train_epochs', 'stft_n_fft', 'wavelet_name', 'wavelet_level', 'wavelet_mode', 'auxi_mode']
metric_names = ['mse', 'mae']

df = []
for exp_dir in exp_dirs:
    runned, setting_dir = exist_metric(exp_dir)
    if not runned:
        continue

    config = load_yaml_as_df(os.path.join(setting_dir, 'config.yaml'))
    metric = load_npy(os.path.join(setting_dir, 'metrics.npy'))
    result = config[params]
    result.loc[:, metric_names] = metric[1], metric[0]
    df.append(result)

df = pd.concat(df, ignore_index=True)
df.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

df.head(4)

,model,seq_len,pred_len,data_id,learning_rate,rec_lambda,auxi_lambda,pca_dim,reinit,use_weights,rank_ratio,auxi_loss,batch_size,lradj,patience,train_epochs,stft_n_fft,wavelet_name,wavelet_level,wavelet_mode,auxi_mode,mse,mae
30,FreTS,96,96,Weather,0.0005,0.8,0.2,T,0,0,1.0,None,32,type1,3,10,64,haar,2,zero,wavelet,0.172263,0.219788
57,FreTS,96,96,Weather,0.0005,0.0,1.0,T,0,0,1.0,None,32,type1,3,10,64,haar,7,zero,wavelet,0.170567,0.216100
67,FreTS,96,96,Weather,0.0005,0.8,0.2,T,0,0,1.0,None,32,type1,3,10,64,haar,3,zero,wavelet,0.171597,0.218063
79,FreTS,96,96,Weather,0.0002,0.4,0.6,T,0,0,1.0,None,32,type1,3,10,64,haar,5,zero,wavelet,0.175353,0.218368


## load data rebb

## preprocess

In [3]:
save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'

baselines = pd.read_csv(f'{save_root}/baselines_params.csv')
finetunes_all = pd.read_csv(f'{save_root}/finetune_all_results.csv')

base = baselines.copy()
base = base[
    ((base.data_id == 'ETTh1') & (base.model == 'Fredformer')) |
    ((base.data_id == 'ECL') & (base.model == 'iTransformer')) |
    ((base.data_id == 'Weather') & (base.model == 'FreTS'))
]

best = finetunes_all.copy()
best = best[
    (best.pca_dim == 'T') &
    (best.use_weights == 0) &
    (best.auxi_loss == 'MAE') &
    (best.reinit == 1) &
    (best.seq_len == 96)
]
best = best[
    ((best.data_id == 'ETTh1_PCA') & (best.model == 'Fredformer')) |
    ((best.data_id == 'ECL_PCA') & (best.model == 'iTransformer')) |
    ((best.data_id == 'Weather_PCA') & (best.model == 'FreTS'))
]
best.rename(columns={'auxi_lambda': 'alpha'}, inplace=True)
best.drop(columns=['rec_lambda'], inplace=True)

trans = df.copy()
trans.rename(columns={'auxi_lambda': 'alpha'}, inplace=True)
trans.drop(columns=['rec_lambda'], inplace=True)


## analysis base

In [4]:
df1 = base.copy()

df1 = df1[['model', 'pred_len', 'data_id', 'mse', 'mae']]

dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Traffic', 'Weather', 'PEMS03', 'PEMS08']
df1['data_id'] = pd.Categorical(df1['data_id'], categories=dst_order, ordered=True)

model_order = ['Fredformer', 'FBM_L', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE', 'DLinear', 'DLinear_Ind', 'FEDformer', 'Autoformer', 'Transformer', 'TCN']
df1['model'] = pd.Categorical(df1['model'], categories=model_order, ordered=True)

df1_avg = df1.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
df1_avg['pred_len'] = 'Avg'
df1 = pd.concat([df1, df1_avg]).reset_index(drop=True)

df1.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

df1.dropna(inplace=True, thresh=4)

df1['label'] = 'DF'
df1

/tmp/ipykernel_1690124/1171081831.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df1_avg = df1.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()


,model,pred_len,data_id,mse,mae,label
4,Fredformer,96,ETTh1,0.377193,0.395888,DF
5,Fredformer,192,ETTh1,0.437019,0.425390,DF
6,Fredformer,336,ETTh1,0.485769,0.448580,DF
7,Fredformer,720,ETTh1,0.487772,0.467352,DF
14,Fredformer,Avg,ETTh1,0.446938,0.434302,DF
8,iTransformer,96,ECL,0.150035,0.241510,DF
9,iTransformer,192,ECL,0.168114,0.259067,DF
10,iTransformer,336,ECL,0.182346,0.274355,DF
11,iTransformer,720,ECL,0.214451,0.303504,DF
34,iTransformer,Avg,ECL,0.178736,0.269609,DF


## analysis trans

In [59]:
columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'alpha', 'stft_n_fft', 'wavelet_name', 'wavelet_level', 'wavelet_mode', 'auxi_mode']

# trans[(trans['pred_len'] == 720) & (trans['data_id'] == 'ETTh1') & (trans['auxi_mode'] == 'stft')].sort_values(by=['pred_len', 'mse'])[columns].head(20)
trans[(trans['pred_len'] == 720) & (trans['data_id'] == 'ETTh1') & (trans['auxi_mode'] == 'wavelet')].sort_values(by=['pred_len', 'mse'])[columns].head(40)
# trans[(trans['pred_len'] == 720) & (trans['data_id'] == 'ECL') & (trans['auxi_mode'] == 'stft')].sort_values(by=['pred_len', 'mse'])[columns].head(15)
# trans[(trans['pred_len'] == 720) & (trans['data_id'] == 'ECL') & (trans['auxi_mode'] == 'wavelet')].sort_values(by=['pred_len', 'mse'])[columns].head(15)
# trans[(trans['pred_len'] == 720) & (trans['data_id'] == 'Weather') & (trans['auxi_mode'] == 'stft')].sort_values(by=['pred_len', 'mse'])[columns].head(15)
# trans[(trans['pred_len'] == 720) & (trans['data_id'] == 'Weather') & (trans['auxi_mode'] == 'wavelet')].sort_values(by=['pred_len', 'mse'])[columns].tail(10)

,model,pred_len,data_id,mse,mae,learning_rate,alpha,stft_n_fft,wavelet_name,wavelet_level,wavelet_mode,auxi_mode
653,Fredformer,720,ETTh1,0.462000,0.460896,0.0010,1.0,64,haar,7,zero,wavelet
915,Fredformer,720,ETTh1,0.462675,0.460819,0.0010,0.8,64,haar,7,zero,wavelet
1219,Fredformer,720,ETTh1,0.462864,0.460540,0.0005,0.4,64,haar,7,zero,wavelet
157,Fredformer,720,ETTh1,0.463262,0.460838,0.0005,0.6,64,haar,7,zero,wavelet
44,Fredformer,720,ETTh1,0.464649,0.460616,0.0005,0.2,64,haar,7,zero,wavelet
1005,Fredformer,720,ETTh1,0.465202,0.461585,0.0005,1.0,64,haar,7,zero,wavelet
918,Fredformer,720,ETTh1,0.465410,0.461546,0.0005,0.8,64,haar,7,zero,wavelet
865,Fredformer,720,ETTh1,0.466102,0.461893,0.0010,0.4,64,haar,7,zero,wavelet
921,Fredformer,720,ETTh1,0.467183,0.456710,0.0005,0.8,64,haar,2,zero,wavelet
1145,Fredformer,720,ETTh1,0.467504,0.462474,0.0010,0.6,64,haar,7,zero,wavelet


In [46]:
df2 = trans.copy()
df2[(df2['pred_len'] == 96) & (df2['data_id'] == 'Weather') & (df2['auxi_mode'] == 'wavelet') & (df2['wavelet_level'] == 3) & (df2.learning_rate == 0.001) & (df2.alpha == 0.2)]

,model,seq_len,pred_len,data_id,learning_rate,alpha,pca_dim,reinit,use_weights,rank_ratio,auxi_loss,batch_size,lradj,patience,train_epochs,stft_n_fft,wavelet_name,wavelet_level,wavelet_mode,auxi_mode,mse,mae
164,FreTS,96,96,Weather,0.001,0.2,T,0,0,1.0,None,32,type1,3,10,64,haar,3,zero,wavelet,0.172843,0.220426


In [60]:
min_mode = 'each'

df2 = trans.copy()

df2_h1_stft_96 = df2[(df2['pred_len'] == 96) & (df2['data_id'] == 'ETTh1') & (df2['auxi_mode'] == 'stft')]
df2_h1_stft_192 = df2[(df2['pred_len'] == 192) & (df2['data_id'] == 'ETTh1') & (df2['auxi_mode'] == 'stft') & (df2['stft_n_fft'] == 32) & (df2.learning_rate == 0.001) & (df2.alpha == 0.8)]
df2_h1_stft_336 = df2[(df2['pred_len'] == 336) & (df2['data_id'] == 'ETTh1') & (df2['auxi_mode'] == 'stft') & (df2['stft_n_fft'] == 128) & (df2.learning_rate == 0.0005) & (df2.alpha == 1.0)]
df2_h1_stft_720 = df2[(df2['pred_len'] == 720) & (df2['data_id'] == 'ETTh1') & (df2['auxi_mode'] == 'stft') & (df2['stft_n_fft'] == 32) & (df2.learning_rate == 0.0005) & (df2.alpha == 0.6)]

df2_h1_wavelet_96 = df2[(df2['pred_len'] == 96) & (df2['data_id'] == 'ETTh1') & (df2['auxi_mode'] == 'wavelet')]
df2_h1_wavelet_192 = df2[(df2['pred_len'] == 192) & (df2['data_id'] == 'ETTh1') & (df2['auxi_mode'] == 'wavelet') & (df2['wavelet_level'] == 5) & (df2.learning_rate == 0.0002) & (df2.alpha == 0.6)]
df2_h1_wavelet_336 = df2[(df2['pred_len'] == 336) & (df2['data_id'] == 'ETTh1') & (df2['auxi_mode'] == 'wavelet')]
df2_h1_wavelet_720 = df2[(df2['pred_len'] == 720) & (df2['data_id'] == 'ETTh1') & (df2['auxi_mode'] == 'wavelet') & (df2['wavelet_level'] == 7) & (df2.learning_rate == 0.0002) & (df2.alpha == 0.2)]

df2_ecl_stft_96 = df2[(df2['pred_len'] == 96) & (df2['data_id'] == 'ECL') & (df2['auxi_mode'] == 'stft') & (df2['stft_n_fft'] == 32) & (df2.learning_rate == 0.001) & (df2.alpha == 0.2)]
df2_ecl_stft_192 = df2[(df2['pred_len'] == 192) & (df2['data_id'] == 'ECL') & (df2['auxi_mode'] == 'stft') & (df2['stft_n_fft'] == 128) & (df2.learning_rate == 0.001) & (df2.alpha == 0.2)]
df2_ecl_stft_336 = df2[(df2['pred_len'] == 336) & (df2['data_id'] == 'ECL') & (df2['auxi_mode'] == 'stft') & (df2['stft_n_fft'] == 32) & (df2.learning_rate == 0.001) & (df2.alpha == 0.6)]
df2_ecl_stft_720 = df2[(df2['pred_len'] == 720) & (df2['data_id'] == 'ECL') & (df2['auxi_mode'] == 'stft') & (df2['stft_n_fft'] == 64) & (df2.learning_rate == 0.0005) & (df2.alpha == 0.6)]

df2_ecl_wavelet_96 = df2[(df2['pred_len'] == 96) & (df2['data_id'] == 'ECL') & (df2['auxi_mode'] == 'wavelet') & (df2['wavelet_level'] == 5) & (df2.learning_rate == 0.001) & (df2.alpha == 0.2)]
df2_ecl_wavelet_192 = df2[(df2['pred_len'] == 192) & (df2['data_id'] == 'ECL') & (df2['auxi_mode'] == 'wavelet') & (df2['wavelet_level'] == 2) & (df2.learning_rate == 0.001) & (df2.alpha == 0.4)]
df2_ecl_wavelet_336 = df2[(df2['pred_len'] == 336) & (df2['data_id'] == 'ECL') & (df2['auxi_mode'] == 'wavelet') & (df2['wavelet_level'] == 3) & (df2.learning_rate == 0.001) & (df2.alpha == 0.4)]
df2_ecl_wavelet_720 = df2[(df2['pred_len'] == 720) & (df2['data_id'] == 'ECL') & (df2['auxi_mode'] == 'wavelet') & (df2['wavelet_level'] == 2) & (df2.learning_rate == 0.001) & (df2.alpha == 0.2)]


df2_wea_stft_96 = df2[(df2['pred_len'] == 96) & (df2['data_id'] == 'Weather') & (df2['auxi_mode'] == 'stft') & (df2['stft_n_fft'] == 128) & (df2.learning_rate == 0.001) & (df2.alpha == 0.6)]
df2_wea_stft_192 = df2[(df2['pred_len'] == 192) & (df2['data_id'] == 'Weather') & (df2['auxi_mode'] == 'stft') & (df2['stft_n_fft'] == 128) & (df2.learning_rate == 0.001) & (df2.alpha == 0.2)]
df2_wea_stft_336 = df2[(df2['pred_len'] == 336) & (df2['data_id'] == 'Weather') & (df2['auxi_mode'] == 'stft') & (df2['stft_n_fft'] == 32) & (df2.learning_rate == 0.001) & (df2.alpha == 0.6)]
df2_wea_stft_720 = df2[(df2['pred_len'] == 720) & (df2['data_id'] == 'Weather') & (df2['auxi_mode'] == 'stft') & (df2['stft_n_fft'] == 128) & (df2.learning_rate == 0.001) & (df2.alpha == 0.6)]

df2_wea_wavelet_96 = df2[(df2['pred_len'] == 96) & (df2['data_id'] == 'Weather') & (df2['auxi_mode'] == 'wavelet') & (df2['wavelet_level'] == 3) & (df2.learning_rate == 0.001) & (df2.alpha == 0.2)]
df2_wea_wavelet_192 = df2[(df2['pred_len'] == 192) & (df2['data_id'] == 'Weather') & (df2['auxi_mode'] == 'wavelet') & (df2['wavelet_level'] == 3) & (df2.learning_rate == 0.001) & (df2.alpha == 0.2)]
df2_wea_wavelet_336 = df2[(df2['pred_len'] == 336) & (df2['data_id'] == 'Weather') & (df2['auxi_mode'] == 'wavelet') & (df2['wavelet_level'] == 5) & (df2.learning_rate == 0.001) & (df2.alpha == 0.2)]
df2_wea_wavelet_720 = df2[(df2['pred_len'] == 720) & (df2['data_id'] == 'Weather') & (df2['auxi_mode'] == 'wavelet') & (df2['wavelet_level'] == 7) & (df2.learning_rate == 0.0002) & (df2.alpha == 1.0)]


df2 = pd.concat([
    df2_h1_stft_96, df2_h1_stft_192, df2_h1_stft_336, df2_h1_stft_720,
    df2_h1_wavelet_96, df2_h1_wavelet_192, df2_h1_wavelet_336, df2_h1_wavelet_720,
    df2_ecl_stft_96, df2_ecl_stft_192, df2_ecl_stft_336, df2_ecl_stft_720,
    df2_ecl_wavelet_96, df2_ecl_wavelet_192, df2_ecl_wavelet_336, df2_ecl_wavelet_720,
    df2_wea_stft_96, df2_wea_stft_192, df2_wea_stft_336, df2_wea_stft_720,
    df2_wea_wavelet_96, df2_wea_wavelet_192, df2_wea_wavelet_336, df2_wea_wavelet_720
], ignore_index=True)


columns = ['model', 'data_id', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights', 'stft_n_fft', 'wavelet_name', 'wavelet_level', 'wavelet_mode', 'auxi_mode']
if min_mode == 'group':
    # df2 = df2.groupby(columns).filter(is_full_group)
    mse_mean = df2.groupby(columns)['mse'].mean().reset_index()
    idx = mse_mean.groupby(['model', 'data_id'])['mse'].idxmin()
    best_model = mse_mean.loc[idx]
    df2 = df2.merge(best_model[columns], on=columns, how='inner')
elif min_mode == 'each':
    min_mse_idx = df2.groupby(['model', 'data_id', 'pred_len', 'auxi_mode'])['mse'].idxmin()
    df2 = df2.loc[min_mse_idx]

columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights', 'stft_n_fft', 'wavelet_name', 'wavelet_level', 'wavelet_mode', 'auxi_mode']
df2 = df2[columns]

dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Traffic', 'Weather', 'PEMS03', 'PEMS08']
df2['data_id'] = pd.Categorical(df2['data_id'], categories=dst_order, ordered=True)

model_order = ['Fredformer', 'FBM_L', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE', 'DLinear', 'DLinear_Ind', 'FEDformer', 'Autoformer', 'Transformer', 'TCN']
df2['model'] = pd.Categorical(df2['model'], categories=model_order, ordered=True)

df2_avg = df2.groupby(['model', 'data_id', 'auxi_mode']).mean(numeric_only=True).reset_index()
df2_avg['pred_len'] = 'Avg'

df2 = pd.concat([df2, df2_avg]).reset_index(drop=True)
df2.sort_values(by=['data_id', 'model', 'auxi_mode', 'pred_len'], inplace=True)

df2.dropna(inplace=True, thresh=5)

df2 = df2[['model', 'pred_len', 'data_id', 'mse', 'mae', 'auxi_mode']]
df2.rename(columns={'auxi_mode': 'label'}, inplace=True)
df2.replace({'label': {'stft': 'STFT', 'wavelet': 'Wavelet'}}, inplace=True)
df2

/tmp/ipykernel_1690124/1645393231.py:67: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df2_avg = df2.groupby(['model', 'data_id', 'auxi_mode']).mean(numeric_only=True).reset_index()


,model,pred_len,data_id,mse,mae,label
8,Fredformer,96,ETTh1,0.371239,0.392394,STFT
10,Fredformer,192,ETTh1,0.428864,0.423183,STFT
12,Fredformer,336,ETTh1,0.476516,0.442619,STFT
14,Fredformer,720,ETTh1,0.475706,0.465915,STFT
28,Fredformer,Avg,ETTh1,0.438081,0.431028,STFT
9,Fredformer,96,ETTh1,0.371383,0.391016,Wavelet
11,Fredformer,192,ETTh1,0.429942,0.424166,Wavelet
13,Fredformer,336,ETTh1,0.477261,0.450060,Wavelet
15,Fredformer,720,ETTh1,0.477289,0.469036,Wavelet
29,Fredformer,Avg,ETTh1,0.438969,0.433569,Wavelet


## analysis ablation 3

In [11]:
min_mode = 'each'

df4 = best.copy()

columns = ['model', 'data_id', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights']
if min_mode == 'group':
    df4 = df4.groupby(columns).filter(is_full_group)
    mse_mean = df4.groupby(columns)['mse'].mean().reset_index()
    idx = mse_mean.groupby(['model', 'data_id'])['mse'].idxmin()
    best_model = mse_mean.loc[idx]
    df4 = df4.merge(best_model[columns], on=columns, how='inner')
elif min_mode == 'each':
    min_mse_idx = df4.groupby(['model', 'data_id', 'pred_len'])['mse'].idxmin()
    df4 = df4.loc[min_mse_idx]

columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights']
df4 = df4[columns]

dst_order = ['ETTm1_PCA', 'ETTm2_PCA', 'ETTh1_PCA', 'ETTh2_PCA', 'ECL_PCA', 'Traffic_PCA', 'Weather_PCA', 'PEMS03_PCA', 'PEMS08_PCA']
df4['data_id'] = pd.Categorical(df4['data_id'], categories=dst_order, ordered=True)

model_order = ['Fredformer', 'FBM_L', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE', 'DLinear', 'DLinear_Ind', 'FEDformer', 'Autoformer', 'Transformer', 'TCN']
df4['model'] = pd.Categorical(df4['model'], categories=model_order, ordered=True)

df4_avg = df4.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
df4_avg['pred_len'] = 'Avg'

df4 = pd.concat([df4, df4_avg]).reset_index(drop=True)
df4.sort_values(by=['data_id', 'model', 'pred_len'], inplace=True)

df4.dropna(inplace=True, thresh=5)

df4['data_id'] = df4['data_id'].str.replace('_PCA', '', regex=False)
df4 = df4[['model', 'pred_len', 'data_id', 'mse', 'mae']]
df4['label'] = 'TransDF'
df4

/tmp/ipykernel_1690124/2474732089.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df4_avg = df4.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()


,model,pred_len,data_id,mse,mae,label
4,Fredformer,96,ETTh1,0.368003,0.390765,TransDF
5,Fredformer,192,ETTh1,0.424089,0.421955,TransDF
6,Fredformer,336,ETTh1,0.466960,0.441356,TransDF
7,Fredformer,720,ETTh1,0.464932,0.463144,TransDF
14,Fredformer,Avg,ETTh1,0.430996,0.429305,TransDF
8,iTransformer,96,ECL,0.144906,0.234782,TransDF
9,iTransformer,192,ECL,0.158948,0.248666,TransDF
10,iTransformer,336,ECL,0.173076,0.264487,TransDF
11,iTransformer,720,ECL,0.203276,0.292044,TransDF
34,iTransformer,Avg,ECL,0.170051,0.259995,TransDF


## concat analysis

In [61]:
compare_columns = ['pred_len', 'mse', 'mae', 'label']
aba_show = pd.concat([df1, df2[compare_columns], df4[compare_columns]], axis=1)


aba_res = pd.concat([df1, df2, df4], axis=0)
save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'

res1 = aba_res[(aba_res['data_id'].isin(['ETTh1', 'ETTh2', 'ETTm1', 'ETTm2', 'ECL', 'Traffic', 'Weather', 'PEMS03', 'PEMS08']))].copy()
# res1 = aba_res[(aba_res['data_id'].isin(['ETTh2', 'ETTm2', 'Traffic', 'PEMS03', 'PEMS08']))].copy()
dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Traffic', 'Weather', 'PEMS03', 'PEMS08']
res1['data_id'] = pd.Categorical(res1['data_id'], categories=dst_order, ordered=True)

label_order = ['DF', 'STFT', 'Wavelet', 'TransDF']
res1['label'] = pd.Categorical(res1['label'], categories=label_order, ordered=True)

res1.sort_values(by=['label', 'data_id', 'pred_len'], inplace=True)

res1 = res1.set_index(['data_id', 'pred_len', 'label']).unstack('label').swaplevel(axis=1)
columns = []
for label in res1.columns.levels[0]:
    columns.append((label, 'mse'))
    columns.append((label, 'mae'))
res1 = res1[columns]


save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
res1.round(3).to_csv(f'{save_root}/vary_trans_local.csv', float_format='%.3f')

res1


label                   DF                STFT             Wavelet            \
                       mse       mae       mse       mae       mse       mae   
data_id pred_len                                                               
ETTh1   96        0.377193  0.395888  0.371239  0.392394  0.371383  0.391016   
        192       0.437019  0.425390  0.428864  0.423183  0.429942  0.424166   
        336       0.485769  0.448580  0.476516  0.442619  0.477261  0.450060   
        720       0.487772  0.467352  0.475706  0.465915  0.477289  0.469036   
        Avg       0.446938  0.434302  0.438081  0.431028  0.438969  0.433569   
ECL     96        0.150035  0.241510  0.146951  0.237907  0.147170  0.236733   
        192       0.168114  0.259067  0.162820  0.253722  0.162914  0.250823   
        336       0.182346  0.274355  0.176270  0.267703  0.176410  0.266087   
        720       0.214451  0.303504  0.209846  0.297097  0.206400  0.293974   
        Avg       0.178736  0.269609  0.173972  0.264107  0.173223  0.261904   
Weather 96        0.173683  0.227737  0.171727  0.223005  0.172843  0.220426   
        192       0.212846  0.266092  0.211911  0.263694  0.212171  0.259457   
        336       0.270492  0.315915  0.261576  0.300421  0.263902  0.300540   
        720       0.337221  0.362299  0.330158  0.352607  0.331620  0.351502   
        Avg       0.248560  0.293011  0.243843  0.284932  0.245134  0.282981   

label              TransDF            
                       mse       mae  
data_id pred_len                      
ETTh1   96        0.368003  0.390765  
        192       0.424089  0.421955  
        336       0.466960  0.441356  
        720       0.464932  0.463144  
        Avg       0.430996  0.429305  
ECL     96        0.144906  0.234782  
        192       0.158948  0.248666  
        336       0.173076  0.264487  
        720       0.203276  0.292044  
        Avg       0.170051  0.259995  
Weather 96        0.169180  0.218550  
        192       0.210151  0.257542  
        336       0.258569  0.297079  
        720       0.327103  0.348704  
        Avg       0.241251  0.280469

In [62]:
import pandas as pd

def mark_min_second_row(series):
    # series: 一行 (即某个data_id, pred_len) 下所有label的mse/mae
    # 格式化为三位小数字符串
    formatted = series.apply(lambda x: "{:.3f}".format(x))
    # 按原始值排序
    sorted_idx = series.argsort()
    min_idx = sorted_idx[0]
    formatted.iloc[min_idx] = f"\\bst{{{formatted.iloc[min_idx]}}}"
    if len(sorted_idx) > 1:
        second_idx = sorted_idx[1]
        formatted.iloc[second_idx] = f"\\subbst{{{formatted.iloc[second_idx]}}}"
    return formatted

def mark_df_min_second(df):
    marked = df.copy()
    for metric in ['mse', 'mae']:
        for idx in marked.index:
            # 取出这一行所有label的当前metric的值
            labels = [col for col in marked.columns if col[1] == metric]
            vals = marked.loc[idx, labels]
            marked_vals = mark_min_second_row(vals)
            marked.loc[idx, labels] = marked_vals.values
    return marked.reset_index().to_latex(index=False, escape=False)

# 假设 res1 就是你的 pivot 后的 DataFrame
marked_res1 = mark_df_min_second(res1)
# marked_res1.to_csv(f'{save_root}/vary_trans_local_marked.csv', float_format='%.3f')
with open(f'{save_root}/vary_trans_local.tex', 'w') as f:
    f.write(marked_res1)

/tmp/ipykernel_1690124/3931819476.py:9: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  min_idx = sorted_idx[0]
/tmp/ipykernel_1690124/3931819476.py:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  second_idx = sorted_idx[1]
/tmp/ipykernel_1690124/3931819476.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.377' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  marked.loc[idx, labels] = marked_vals.values
/tmp/ipykernel_1690124/3931819476.py:24: FutureWarning: Setting an item of incompatible